# 09 — DTI Filtered Dataset

Create a filtered version of the full DTI 100K dataset (`dti_esm2_100k.csv`) that removes all SMILES present in any ADMET benchmark test set. This prevents data leakage during pre-training → ADMET fine-tuning evaluation.

Same approach as the existing `dti_esm2_10k_filtered.csv`, but applied to the full 100K dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem

DATA_DIR = Path("/home/shpark/prj-molrepr/graphium/data/dti-processed")
OUTPUT_PATH = DATA_DIR / "dti_esm2_100k_filtered.csv"

# Load full DTI dataset
dti_df = pd.read_csv(DATA_DIR / "dti_esm2_100k.csv")
print(f"Full DTI dataset: {len(dti_df)} rows, {len(dti_df.columns)} columns")
print(f"SMILES column: SMILES_nometa")
print(f"Sample: {dti_df['SMILES_nometa'].iloc[0][:60]}")

Full DTI dataset: 100000 rows, 2561 columns
SMILES column: SMILES_nometa
Sample: Cc1ccc(COc2ccc3nc([C@H]4CCCC[C@H]4C(=O)O)n(Cc4ccc(OC(F)(F)F)


## Collect ADMET test set SMILES (all 22 tasks)

Use TDC to get the official train/test splits for all 22 ADMET benchmark tasks. Collect all test SMILES and canonicalize them for matching.

In [2]:
from tdc.benchmark_group import admet_group
import tempfile

ADMET_TASKS = [
    # Absorption
    "caco2_wang", "hia_hou", "pgp_broccatelli", "bioavailability_ma",
    "lipophilicity_astrazeneca", "solubility_aqsoldb", "bbb_martins", "ppbr_az", "vdss_lombardo",
    # Metabolism
    "cyp2d6_veith", "cyp3a4_veith", "cyp2c9_veith",
    "cyp2c9_substrate_carbonmangels", "cyp2d6_substrate_carbonmangels", "cyp3a4_substrate_carbonmangels",
    # Excretion
    "half_life_obach", "clearance_hepatocyte_az", "clearance_microsome_az",
    # Toxicity
    "ld50_zhu", "herg", "ames", "dili",
]


def canonicalize(smi):
    """Canonicalize SMILES using RDKit."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return smi  # Return original if parsing fails
    return Chem.MolToSmiles(mol)


# Collect all ADMET test SMILES
group = admet_group(path=tempfile.mkdtemp())
admet_test_smiles = set()
admet_all_smiles = set()

for task in ADMET_TASKS:
    benchmark = group.get(task)
    test_df = benchmark["test"]
    test_smi = set(test_df["Drug"].apply(canonicalize).tolist())
    admet_test_smiles.update(test_smi)
    
    # Also collect train+val for stats
    train_df, val_df = group.get_train_valid_split(seed=42, benchmark=task)
    all_smi = set(test_df["Drug"].tolist()) | set(train_df["Drug"].tolist()) | set(val_df["Drug"].tolist())
    admet_all_smiles.update(all_smi)
    
    print(f"  {task:35s} test={len(test_smi):5d}  total={len(all_smi):5d}")

print(f"\nTotal unique ADMET test SMILES:  {len(admet_test_smiles)}")
print(f"Total unique ADMET all SMILES:   {len(admet_all_smiles)}")

  0%|          | 0.00/1.47M [00:00<?, ?iB/s]

  1%|          | 17.4k/1.47M [00:00<00:16, 89.5kiB/s]

  4%|▍         | 62.5k/1.47M [00:00<00:08, 173kiB/s] 

  8%|▊         | 124k/1.47M [00:00<00:05, 239kiB/s] 

 19%|█▉        | 281k/1.47M [00:00<00:02, 463kiB/s]

...


generating training, validation splits...


  ames                                test= 1453  total= 7255


  0%|          | 0/379 [00:00<?, ?it/s]

100%|██████████| 379/379 [00:00<00:00, 4912.67it/s]

  dili                                test=   96  total=  475

Total unique ADMET test SMILES:  13818
Total unique ADMET all SMILES:   46202


## Filter DTI dataset

Remove DTI rows whose canonicalized SMILES appear in any ADMET test set.

In [3]:
# Canonicalize DTI SMILES for matching
print("Canonicalizing DTI SMILES...")
dti_df["SMILES_canon"] = dti_df["SMILES_nometa"].apply(canonicalize)

# Check overlap
dti_smiles = set(dti_df["SMILES_canon"].tolist())
overlap_test = dti_smiles & admet_test_smiles
overlap_all = dti_smiles & admet_all_smiles

print(f"\nDTI unique SMILES:           {len(dti_smiles)}")
print(f"Overlap with ADMET test:     {len(overlap_test)}")
print(f"Overlap with ADMET all:      {len(overlap_all)}")
print(f"Overlap % (test):            {len(overlap_test)/len(dti_smiles)*100:.2f}%")

# Filter: remove rows with test set SMILES
mask_keep = ~dti_df["SMILES_canon"].isin(admet_test_smiles)
dti_filtered = dti_df[mask_keep].drop(columns=["SMILES_canon"]).reset_index(drop=True)

print(f"\nFiltered DTI dataset:")
print(f"  Before: {len(dti_df)} rows")
print(f"  Removed: {len(dti_df) - len(dti_filtered)} rows (ADMET test set overlap)")
print(f"  After:  {len(dti_filtered)} rows")

Canonicalizing DTI SMILES...



DTI unique SMILES:           71786
Overlap with ADMET test:     852
Overlap with ADMET all:      2012
Overlap % (test):            1.19%



Filtered DTI dataset:
  Before: 100000 rows
  Removed: 4518 rows (ADMET test set overlap)
  After:  95482 rows


## Save and verify

In [4]:
# Save filtered dataset
dti_filtered.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1e6:.1f} MB")

# Verify: no overlap with ADMET test sets
verify_df = pd.read_csv(OUTPUT_PATH)
verify_smiles = set(verify_df["SMILES_nometa"].apply(canonicalize).tolist())
verify_overlap = verify_smiles & admet_test_smiles
print(f"\nVerification:")
print(f"  Loaded rows:     {len(verify_df)}")
print(f"  Unique SMILES:   {len(verify_smiles)}")
print(f"  ADMET test leak: {len(verify_overlap)} (should be 0)")
assert len(verify_overlap) == 0, f"Data leakage detected: {len(verify_overlap)} SMILES"
print("  PASS: No data leakage")

# Compare with existing 10K filtered
existing_10k = pd.read_csv(DATA_DIR / "dti_esm2_10k_filtered.csv")
print(f"\nComparison:")
print(f"  dti_esm2_10k.csv:           {10000} rows")
print(f"  dti_esm2_10k_filtered.csv:  {len(existing_10k)} rows (removed {10000 - len(existing_10k)})")
print(f"  dti_esm2_100k.csv:          {len(dti_df)} rows")
print(f"  dti_esm2_100k_filtered.csv: {len(dti_filtered)} rows (removed {len(dti_df) - len(dti_filtered)})")

Saved to: /home/shpark/prj-molrepr/graphium/data/dti-processed/dti_esm2_100k_filtered.csv
File size: 2676.3 MB



Verification:
  Loaded rows:     95482
  Unique SMILES:   70934
  ADMET test leak: 0 (should be 0)
  PASS: No data leakage



Comparison:
  dti_esm2_10k.csv:           10000 rows
  dti_esm2_10k_filtered.csv:  9556 rows (removed 444)
  dti_esm2_100k.csv:          100000 rows
  dti_esm2_100k_filtered.csv: 95482 rows (removed 4518)
